# 03 • Tipos, ausências e qualidade

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [1]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [2]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

species    island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g    sex  year
 Adelie Torgersen            39.1           18.7              181.0       3750.0   male  2007
 Adelie Torgersen            39.5           17.4              186.0       3800.0 female  2007
 Adelie Torgersen            40.3           18.0              195.0       3250.0 female  2007
Dimensões: (344, 8)
Colunas: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
Series
   species     island  body_mass_g
7   Adelie  Torgersen       4675.0
9   Adelie  Torgersen       4250.0
14  Adelie  Torgersen       4400.0
17  Adelie  Torgersen       4500.0
19  Adelie  Torgersen       4200.0
  species     island
0  Adelie  Torgersen
1  Adelie  Torgersen
2  Adelie  Torgersen


### Objetivo: distinguir valor conhecido, valor ausente e valor inválido

Uma tabela pode abrir sem erros e ainda conter dados inadequados para uma análise. Nesta aula, vamos responder: “em quais colunas faltam dados?” e “quais linhas posso usar para estudar a massa corporal?”.

### 1. O que significa um valor ausente?


In [3]:
print(p[['species', 'body_mass_g']].head(4))


  species  body_mass_g
0  Adelie       3750.0
1  Adelie       3800.0
2  Adelie       3250.0
3  Adelie          NaN


Na quarta observação, a massa aparece como NaN. Isso indica ausência de uma medição nessa coluna. Não significa que o animal pesa zero e não autoriza inventar sua massa. Em outras colunas ou tipos, a ausência pode aparecer como None, <NA> ou NaT. Use isna para detectar ausências; não tente comparar com o texto 'NaN'.


In [4]:
falta_massa = p['body_mass_g'].isna()
print(falta_massa.head(4))
print(falta_massa.sum())


0    False
1    False
2    False
3     True
Name: body_mass_g, dtype: bool
2


O começo da condição é False, False, False, True. A soma conta True como 1 e False como 0. O total da base é 2: há duas observações sem massa. Isso deixa 342 medições presentes, não 344.

### 2. De contagem a porcentagem


In [5]:
quantidade = falta_massa.sum()
total = len(p)
percentual = quantidade / total * 100
print(quantidade, total, round(percentual, 2))


2 344 0.58


len(p) é a quantidade de linhas. A conta é 2 dividido por 344, multiplicado por 100: aproximadamente 0,58%. round(..., 2) arredonda apenas o número apresentado para duas casas. Outra forma é `falta_massa.mean() * 100`, porque a média dos zeros e uns representa a proporção de ausências.


In [6]:
print(p.isna().sum())


species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64


Aplicado à tabela inteira, isna cria uma tabela de respostas; sum conta por coluna. Nesta base, sex tem 11 ausências e body_mass_g tem 2. Depois de entender isso, podemos reunir contagem e porcentagem em um relatório, como no exemplo guiado ao final.

### 3. Remover somente o que a pergunta exige


In [7]:
com_massa = p.dropna(subset=['body_mass_g']).copy()
print(len(com_massa))


342


dropna remove linhas com ausência nas colunas indicadas em subset. O resultado tem 342 linhas. A chamada copy deixa explícito que trabalharemos com uma tabela separada. A base p permanece com 344 linhas. Não use simplesmente `p.dropna()` sem pensar: isso exige informação presente em todas as colunas e pode descartar animais por falta de sexo, mesmo quando você só quer estudar massa.

Se a pergunta envolver massa e nadadeira, use `subset=['body_mass_g', 'flipper_length_mm']`. Documente a perda de observações. Preencher por média ou mediana é uma escolha metodológica, não uma etapa obrigatória de limpeza; não faremos isso automaticamente.

### 4. Tipos: o que a coluna permite fazer?


In [8]:
print(p.dtypes)


species                  str
island                   str
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                      str
year                   int64
dtype: object


dtypes informa o tipo de cada coluna. O tipo numérico permite fazer contas. Um tipo de texto guarda nomes ou códigos; sua aparência exata pode variar entre versões. Um número que veio como texto precisa ser convertido antes de uma operação numérica. Porém, um código como '0012' deve continuar texto para preservar seus zeros e seu significado. Ter apenas dígitos não transforma um identificador em medida.

Para listar colunas reconhecidas como numéricas, use `p.select_dtypes(include='number').columns`. Essa operação seleciona por tipo, não por significado: ainda cabe a você decidir se faz sentido calcular uma média daquela coluna.

### 5. Converter e contar os problemas


In [9]:
entrada = pd.Series(['10', 'erro', None])
numeros = pd.to_numeric(entrada, errors='coerce')
print(numeros)


0    10.0
1     NaN
2     NaN
dtype: float64


Este é um exemplo fictício de importação problemática. O primeiro valor vira 10. O texto erro não pode virar número. Com errors='coerce', ele vira ausência em vez de interromper a conversão. A ausência original também continua ausente. Ao final há duas ausências, mas só uma foi criada por falha de conversão:


In [10]:
novas_falhas = entrada.notna() & numeros.isna()
print(novas_falhas.sum())


1


notna é o contrário de isna. A condição procura valores que estavam presentes antes e ficaram ausentes depois. O resultado esperado é 1. Usar coerce sem medir esse efeito pode esconder erros importantes. astype é útil quando você já sabe que a conversão deve funcionar, como `.astype('Int64')` para inteiros que aceitam ausências. Datas terão uma aula própria.

### 6. Duplicatas exigem uma definição


In [11]:
print(p.duplicated().sum())


0


duplicated marca linhas iguais a uma anterior, considerando todas as colunas por padrão. O total encontrado não é uma autorização para apagar. Dois animais podem ter as mesmas medidas registradas, e a base simplificada não tem um identificador individual explícito. drop_duplicates remove essas repetições, mas só deve ser usado depois de justificar o que caracteriza um registro duplicado no seu problema.

### O que guardar desta aula

Ausência não é zero; converter não é corrigir; remover exige justificativa. Antes de uma limpeza, registre o tamanho original. Depois, registre quantas linhas foram removidas e por qual motivo. Preserve o arquivo bruto para conseguir revisar suas decisões.


## Exemplo guiado

In [12]:
qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

                   ausentes  percentual
species                   0    0.000000
island                    0    0.000000
bill_length_mm            2    0.581395
bill_depth_mm             2    0.581395
flipper_length_mm         2    0.581395
body_mass_g               2    0.581395
sex                      11    3.197674
year                      0    0.000000
Observações com massa: 342


## Exercício 03.1
Quais colunas possuem pelo menos uma ausência?

In [13]:
n = p.isna().sum(); print(n[n.gt(0)])

bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


## Exercício 03.2
Calcule a porcentagem de massas ausentes.

In [14]:
print(p['body_mass_g'].isna().mean()*100)

0.5813953488372093


## Exercício 03.3
Crie uma tabela apenas com massa e comprimento de nadadeira presentes. Conte a perda de linhas.

In [15]:
completos = p.dropna(subset=['body_mass_g','flipper_length_mm'])
print('Removidas:',len(p)-len(completos))

Removidas: 2


## Exercício 03.4
Converta a Series didática ["10", "erro", None] para número e conte ausências resultantes.

In [16]:
s = pd.Series(['10','erro',None]); n = pd.to_numeric(s, errors='coerce')
print(n); print(n.isna().sum())
# Uma falha de conversão e uma ausência original.

0    10.0
1     NaN
2     NaN
dtype: float64
2


## Exercício 03.5
Conte duplicatas exatas, sem removê-las. Por que isso não basta para excluir observações?

In [17]:
print(p.duplicated().sum())
# Sem um identificador individual, medições iguais não provam duplicação indevida.

0
